# NDVI Forest Timelapse — Walkthrough

Build an annotated Sentinel-2 **NDVI timelapse** (MP4 + GIF) over a forest area
of interest, step by step, using the `gee_animation` package.

Each stage runs one part of the pipeline and shows its intermediate output:
authenticate → parameters → AOI → cloud-masked NDVI collection → monthly median
frames → preview one frame → render the animation → display it inline.

**Prerequisites** (run once, in a terminal):

```bash
pip install -e ".[notebook]"
earthengine authenticate      # one-time Google Earth Engine login
```

> This notebook makes **live Earth Engine calls** — it needs network access and
> a configured EE project (`hnee-331218` by default).

## 1. Imports & authenticate

In [ ]:
from gee_animation import auth, aoi, collection, compositing, render
from gee_animation.imaging import colorize
from gee_animation.config import RunConfig

auth.init("hnee-331218")   # ee.Initialize(project=...); prompts to auth if needed
print("Earth Engine initialised.")

## 2. Parameters

Everything about a run lives in a `RunConfig`. Edit any field and re-run the
notebook from here down. The NDVI `min`/`max` and `palette` are applied
identically to every frame so colour is comparable across the animation.

In [ ]:
cfg = RunConfig(
    name="barnim_ndvi",                         # output basename -> out/barnim_ndvi.{mp4,gif}
    project="hnee-331218",                       # Earth Engine project
    aoi={"bbox": [13.7, 52.8, 13.9, 52.95]},     # [minLon, minLat, maxLon, maxLat]
    start="2022-01-01",                          # inclusive
    end="2023-01-01",                            # exclusive
    sensor="sentinel2",
    cadence="monthly",
    max_cloud_percent=60,                        # scene-level pre-filter
    ndvi_min=-0.2,                               # fixed NDVI colour scale...
    ndvi_max=0.9,                                # ...comparable across frames
    palette=["#a1622f", "#e8d9a0", "#3b7a2a"],   # brown -> tan -> green
    fps=4,
    scale=20,                                    # informational; dimensions drives size
    dimensions=768,                              # max width/height of each frame
    out_dir="out",
)
cfg

## 3. Area of interest

Turn the bbox (or a GeoJSON path) into an Earth Engine geometry.

In [ ]:
geom = aoi.parse(cfg.aoi)
print("AOI bounds:", geom.bounds().getInfo()["coordinates"])

## 4. Build the cloud-masked NDVI collection

Filter Sentinel-2 (`COPERNICUS/S2_SR_HARMONIZED`) by date, bounds and cloud
percentage, mask clouds/shadows/snow via the SCL band, and add an `NDVI` band.

In [ ]:
coll = collection.build(cfg, geom)
n_scenes = coll.size().getInfo()
print(f"{n_scenes} Sentinel-2 scenes after date / bounds / cloud filtering")

## 5. Monthly median frames

Reduce each month to a cloud-robust median NDVI image. Months with no imagery
are skipped, so the frame count may be fewer than the number of months.

In [ ]:
frames = compositing.monthly_median(coll, cfg)
print(f"{len(frames)} monthly frames:")
print(", ".join(f.label for f in frames))

## 6. Preview a single frame

Render just the first frame so you can check the AOI, palette and cloud masking
before committing to the full animation. Cloud/no-data pixels are painted a
neutral grey (not a vegetation colour).

In [ ]:
from PIL import Image
from IPython.display import display

frame = frames[0]
# render._fetch_thumbnail is the package's internal EE download; used here to
# preview one frame with the same styling the full render applies.
ndvi_arr, valid = render._fetch_thumbnail(frame.image, cfg, geom)
rgb = colorize(ndvi_arr, cfg.ndvi_min, cfg.ndvi_max, cfg.palette)
rgb = render.apply_nodata(rgb, valid)     # cloud / no-data -> neutral grey
rgb = render.annotate(rgb, frame.label)   # month label
rgb = render.add_colorbar(rgb, cfg)       # shared NDVI colorbar
print(f"Preview of {frame.label}:")
display(Image.fromarray(rgb))

## 7. Render the animation

Run the full pipeline over every frame and write `out/<name>.mp4` and
`out/<name>.gif`. (Falls back to GIF-only if MP4 encoding is unavailable.)

In [ ]:
paths = render.render(frames, cfg, geometry=geom)
paths

## 8. Show the result

In [ ]:
from IPython.display import Image as IPyImage

gif = next(p for p in paths if p.suffix == ".gif")
print(f"{gif}")
IPyImage(filename=str(gif))

In [ ]:
from IPython.display import Video

mp4 = next((p for p in paths if p.suffix == ".mp4"), None)
Video(str(mp4), embed=True) if mp4 else "MP4 not produced (ffmpeg unavailable) - see GIF above."